<a href="https://colab.research.google.com/github/ever-oli/TensorPoly/blob/main/TensorPoly/R/UNet_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
unet_encoder_block <- function(x, out_channels) {
  # x: assumed to be a 4D array (batch, H, W, in_channels)
  # out_channels: integer

  # Extract input dimensions
  # dim(x) returns a vector of dimensions. Assuming the order is (batch, H, W, in_channels)
  dimensions <- dim(x)
  batch <- dimensions[1]
  H <- dimensions[2]
  W <- dimensions[3]
  in_channels <- dimensions[4] # Not directly used in the calculations below, but kept for context

  # After two 3x3 convolutions with no padding:
  # Each conv reduces height and width by 2
  # conv1: (H, W) -> (H-2, W-2)
  # conv2: (H-2, W-2) -> (H-4, W-4)
  skip_H <- H - 4
  skip_W <- W - 4

  # Skip connection features (after conv2, before pooling)
  # In R, to create an array of zeros: array(0, dim = c(...))
  skip_out <- array(0, dim = c(batch, skip_H, skip_W, out_channels))

  # After 2x2 max pooling with stride 2:
  # Height and width are halved
  # For integer division in R, use floor(x / y)
  pool_H <- floor(skip_H / 2)
  pool_W <- floor(skip_W / 2)

  # Pooled output (after max pooling)
  pool_out <- array(0, dim = c(batch, pool_H, pool_W, out_channels))

  # Return a list of the two output arrays (R's equivalent of a Python tuple for multiple returns)
  return(list(pool_out = pool_out, skip_out = skip_out))
}

In [2]:
unet_decoder_block <- function(x, skip, out_channels) {
  # x: assumed to be a 4D array (batch, H, W, in_channels) from the previous encoder block
  # skip: assumed to be a 4D array (batch, H_skip, W_skip, skip_channels) from the corresponding encoder skip connection
  # out_channels: integer, the number of output channels after the double convolution

  # Extract dimensions from the input feature map (x)
  dimensions_x <- dim(x)
  batch <- dimensions_x[1]
  H <- dimensions_x[2]
  W <- dimensions_x[3]
  in_channels <- dimensions_x[4] # Not directly used in calculations below, but kept for context

  # Extract dimensions from the skip connection (skip)
  dimensions_skip <- dim(skip)
  batch_skip <- dimensions_skip[1] # Should be equal to 'batch'
  H_skip <- dimensions_skip[2]
  W_skip <- dimensions_skip[3]
  skip_channels <- dimensions_skip[4]

  # Step 1: Upsample - double the spatial dimensions
  # In a real implementation, this would involve a transpose convolution or upsampling layer.
  H_up <- H * 2
  W_up <- W * 2

  # Step 2: Crop skip connection to match upsampled size
  # Calculate crop margins (center crop) for height and width
  # R's integer division: floor(x / y)
  crop_h <- floor((H_skip - H_up) / 2)
  crop_w <- floor((W_skip - W_up) / 2)

  # Cropped skip connection
  # R uses 1-based indexing. The slice `start:end` is inclusive.
  # For a 4D array [batch, H, W, channels], we slice H and W dimensions.
  # 'drop=FALSE' ensures the dimensions are not dropped if a slice results in a single element.
  skip_cropped <- skip[1:batch_skip, (crop_h + 1):(crop_h + H_up), (crop_w + 1):(crop_w + W_up), 1:skip_channels, drop = FALSE]

  # In a real implementation, 'x' would also be upsampled and then concatenated with 'skip_cropped'.
  # The output shape of the upsampled 'x' would be (batch, H_up, W_up, in_channels).
  # After concatenation along the channel axis:
  # (batch, H_up, W_up, upsampled_x_channels + skip_cropped_channels)
  # Here, we're assuming the upsampled x has `in_channels` and `skip_cropped` has `skip_channels`.
  # So, the combined channels would be `in_channels + skip_channels` if x was upsampled to match skip_cropped's channel count,
  # but the problem implies the `out_channels` parameter is for the final output of the block.

  # Step 4: Two 3x3 unpadded convolutions
  # Each convolution reduces H and W by 2. Total reduction is 4.
  # This assumes the combined feature map (after upsampling x and concatenating with skip_cropped)
  # has spatial dimensions H_up and W_up.
  H_out <- H_up - 4
  W_out <- W_up - 4

  # Create output array with the calculated shape and desired out_channels
  # In R, array(value, dim = c(d1, d2, d3, d4))
  output <- array(0, dim = c(batch, H_out, W_out, out_channels))

  return(output)
}

In [3]:
crop_and_concat <- function(encoder_features, decoder_features) {
  # encoder_features: assumed to be a 4D array (batch, H_enc, W_enc, C_enc)
  # decoder_features: assumed to be a 4D array (batch, H_dec, W_dec, C_dec)

  # Extract dimensions from encoder_features
  dimensions_enc <- dim(encoder_features)
  batch <- dimensions_enc[1]
  H_enc <- dimensions_enc[2]
  W_enc <- dimensions_enc[3]
  C_enc <- dimensions_enc[4]

  # Extract dimensions from decoder_features
  dimensions_dec <- dim(decoder_features)
  # Assuming batch is the same for both
  H_dec <- dimensions_dec[2]
  W_dec <- dimensions_dec[3]
  C_dec <- dimensions_dec[4]

  # Calculate crop margins for center cropping
  # R's integer division: floor(x / y)
  crop_h <- floor((H_enc - H_dec) / 2)
  crop_w <- floor((W_enc - W_dec) / 2)

  # Perform center crop on encoder features
  # R uses 1-based indexing. The slice `start:end` is inclusive.
  # For a 4D array [batch, H, W, channels], we slice H and W dimensions.
  # 'drop=FALSE' ensures the dimensions are not dropped if a slice results in a single element.
  encoder_cropped <- encoder_features[
    1:batch,
    (crop_h + 1):(crop_h + H_dec),
    (crop_w + 1):(crop_w + W_dec),
    1:C_enc,
    drop = FALSE
  ]

  # Concatenate along channel dimension (axis=-1 in NumPy, which is the 4th dimension in R)
  # In a real deep learning scenario in R, this would often involve a dedicated tensor library
  # like 'torch' or 'tensorflow', or the 'abind' package for array binding.
  # For this conceptual translation, we'll create a new array with combined channels.
  output_channels <- C_enc + C_dec
  output <- array(0, dim = c(batch, H_dec, W_dec, output_channels))

  # Note: Actual concatenation logic (e.g., copying data from encoder_cropped and decoder_features
  # into 'output') is omitted as per the pattern established in previous translations,
  # focusing on dimension calculation and placeholder output.

  return(output)
}

In [4]:
unet_bottleneck <- function(x, out_channels) {
  # x: assumed to be a 4D array (batch, H, W, in_channels)
  # out_channels: integer

  # Extract input dimensions
  # dim(x) returns a vector of dimensions. Assuming the order is (batch, H, W, in_channels)
  dimensions <- dim(x)
  batch <- dimensions[1]
  H <- dimensions[2]
  W <- dimensions[3]
  in_channels <- dimensions[4] # Not directly used in calculations below, but kept for context

  # Two 3x3 unpadded convolutions reduce spatial dimensions by 4 total
  # Each conv reduces height and width by 2
  # conv1: (H, W) -> (H-2, W-2)
  # conv2: (H-2, W-2) -> (H-4, W-4)
  H_out <- H - 4
  W_out <- W - 4

  # Create output array with the calculated shape and desired out_channels
  # In R, array(value, dim = c(d1, d2, d3, d4))
  output <- array(0, dim = c(batch, H_out, W_out, out_channels))

  return(output)
}

In [5]:
unet_output <- function(features, num_classes) {
  # features: assumed to be a 4D array (batch, H, W, feat_channels)
  # num_classes: integer, the number of output classes

  # Extract dimensions
  # dim(features) returns a vector of dimensions. Assuming the order is (batch, H, W, feat_channels)
  dimensions <- dim(features)
  batch <- dimensions[1]
  H <- dimensions[2]
  W <- dimensions[3]
  feat_channels <- dimensions[4] # Not directly used in calculations below, but kept for context

  # 1x1 convolution preserves spatial dimensions
  # Output channels = num_classes
  # In R, to create an array of zeros: array(0, dim = c(...))
  output <- array(0, dim = c(batch, H, W, num_classes))

  return(output)
}

In [6]:
unet <- function(x, num_classes = 2) {
  # x: assumed to be a 4D array (batch, H, W, in_channels)
  # num_classes: integer, the number of output classes

  # =========== ENCODER PATH ===========
  # Store skip connections for decoder

  # Encoder 1: 64 channels
  e1_results <- unet_encoder_block(x, out_channels = 64)
  e1_pool <- e1_results$pool_out
  e1_skip <- e1_results$skip_out

  # Encoder 2: 128 channels
  e2_results <- unet_encoder_block(e1_pool, out_channels = 128)
  e2_pool <- e2_results$pool_out
  e2_skip <- e2_results$skip_out

  # Encoder 3: 256 channels
  e3_results <- unet_encoder_block(e2_pool, out_channels = 256)
  e3_pool <- e3_results$pool_out
  e3_skip <- e3_results$skip_out

  # Encoder 4: 512 channels
  e4_results <- unet_encoder_block(e3_pool, out_channels = 512)
  e4_pool <- e4_results$pool_out
  e4_skip <- e4_results$skip_out

  # =========== BOTTLENECK ===========
  # Deepest layer: 1024 channels
  bottleneck_out <- unet_bottleneck(e4_pool, out_channels = 1024)

  # =========== DECODER PATH ===========
  # Use skips in reverse order (last encoder skip goes to first decoder)

  # Decoder 4: 512 channels (using e4_skip)
  d4_out <- unet_decoder_block(bottleneck_out, e4_skip, out_channels = 512)

  # Decoder 3: 256 channels (using e3_skip)
  d3_out <- unet_decoder_block(d4_out, e3_skip, out_channels = 256)

  # Decoder 2: 128 channels (using e2_skip)
  d2_out <- unet_decoder_block(d3_out, e2_skip, out_channels = 128)

  # Decoder 1: 64 channels (using e1_skip)
  d1_out <- unet_decoder_block(d2_out, e1_skip, out_channels = 64)

  # =========== OUTPUT LAYER ===========
  # Final 1x1 convolution to get class scores
  output <- unet_output(d1_out, num_classes)

  return(output)
}